# Deep Learning for Computer Vision: From Classification to Segmentation

## BMVA Summer School : Beginner Session

<a href="https://colab.research.google.com/github/Adaptive-Intelligence-Lab/BMVA-summer-school/blob/main/BMVA_CVSS_Beginner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---

| | Practical | What you build |
|---|:---|:---|
| **1** | A Hands-On Introduction to Deep Learning for Computer Vision | your first CNN, trained to classify MNIST handwritten digits |
| **2** | Fully Convolutional Networks for Semantic Segmentation | an FCN that labels *every pixel* of real cat and dog photos |

Practical 1 covers the fundamentals — tensors, convolutions, models, training loops, evaluation. Practical 2 builds directly on them, turning a pretrained ImageNet classifier into a dense, per-pixel segmentation model. Work through them in order, top to bottom.

> **⚠️ Set up your runtime before running any cells.** Practical 2 trains much faster on a GPU, and changing the runtime type later **restarts the session and wipes everything you've run**. Go to **Runtime → Change runtime type → T4 GPU** *now*. (Practical 1 is light and runs happily on the GPU too.)

Each practical keeps its own part numbering (both start at Part 0), and Practical 2 performs its own setup, so the second half also works on its own in a fresh session if you have done Practical 1 before.


# Practical 1 — A Hands-On Introduction to Deep Learning for Computer Vision

## BMVA Summer School — Practical 1 of 2


---

Welcome! In this practical you will build, train and test your first **convolutional neural network (CNN)** — the family of models behind most of modern computer vision — using [PyTorch](https://pytorch.org/).

Our task is a classic: teaching a network to recognise **handwritten digits** (0–9) from the **MNIST** dataset. It is small enough to train in a couple of minutes, yet rich enough to demonstrate every ingredient of real deep learning systems.

### What you will learn

1. **Data** — how images become tensors, and how PyTorch loads them in batches
2. **Convolutions** — what they compute and why they suit images
3. **Models** — how to define a CNN with `torch.nn`
4. **Training** — *everything* you need to train a network: a loss function, an optimiser, and the training loop
5. **Evaluation** — accuracy, confusion matrices, and looking at the mistakes
6. **Exercises** — build a deeper network and experiment with different loss functions

### How to use this notebook

- Run each cell in order with **Shift + Enter**.
- **🧠 Quick check** boxes ask you a short question about what you have just seen — think first, then click to reveal the answer.
- **✏️ Exercises** in Part 6 are where you write code yourself. Solutions are at the very end, but do try first!

(Exercise 3 is an optional extension — if the hour runs out, it makes a good take-home.)

This notebook is self-contained and needs nothing beyond `torch`, `torchvision`, `matplotlib` and `numpy` — all pre-installed on Google Colab. It runs in a few minutes even without a GPU.

> *Inspired by the [dl-pytorch](https://github.com/atapour/dl-pytorch) teaching materials by Amir Atapour-Abarghouei (Durham University).*

---
# Part 0 — Setup

**On Google Colab:** a GPU makes training faster, but is *not required* for this practical. To use one, go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** *before* running any cells.

**Running locally instead?** Install the dependencies with:
`pip install torch torchvision matplotlib numpy`

Let's start by importing what we need:

In [ ]:
import torch                                  # the core library: tensors and automatic differentiation
import torch.nn as nn                         # neural network layers and loss functions
import torch.nn.functional as F               # the same operations as plain functions
from torch.utils.data import DataLoader       # batching and shuffling of datasets

import torchvision
from torchvision import datasets, transforms  # standard vision datasets and image transforms

import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version:     {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

PyTorch code is *device-agnostic*: the same code runs on a CPU, an NVIDIA GPU (`cuda`) or an Apple Silicon GPU (`mps`). We pick the best device available and later move our data and model onto it.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")     # NVIDIA GPU (this is what you get on Colab)
elif torch.backends.mps.is_available():
    device = torch.device("mps")      # Apple Silicon GPU
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Fix the random seed so that everyone gets (roughly) the same results:
torch.manual_seed(42);

---
# Part 1 — The data: MNIST

Deep learning starts with data. **MNIST** contains 70,000 grayscale images of handwritten digits, each **28 × 28 pixels**, split into 60,000 training images and 10,000 test images. Every image comes with a **label**: the digit it shows (0–9).

Why two separate sets?

- The **training set** is what the network learns from.
- The **test set** is kept aside to measure how well the network handles digits it has *never seen*. Testing on training data would be like marking an exam whose questions the student memorised in advance.

`torchvision` downloads MNIST for us. We attach a small pipeline of **transforms** that is applied to every image as it is loaded:

1. `ToTensor()` converts the image to a PyTorch **tensor** (a multi-dimensional array) of shape `[1, 28, 28]` — that is `[channels, height, width]`, with 1 channel because the images are grayscale — and scales pixel values from 0–255 down to 0–1.
2. `Normalize(mean, std)` then standardises the values using MNIST's known mean (0.1307) and standard deviation (0.3081). Inputs centred around zero make training smoother and faster.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                        # image -> tensor [1, 28, 28], values in [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)),   # standardise with MNIST's mean and std
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")

A dataset behaves like a list: indexing it gives one `(image, label)` pair. Let's inspect a single sample:

In [ ]:
image, label = train_dataset[0]

print(f"Image type:  {type(image).__name__}")
print(f"Image shape: {list(image.shape)}   # [channels, height, width]")
print(f"Pixel range: {image.min():.2f} to {image.max():.2f}   # after normalisation")
print(f"Label:       {label}")

Networks are not trained one image at a time but on **mini-batches** — here, 128 images at once. Batches average the gradient over many examples (a more reliable learning signal) and make much better use of parallel hardware.

The **`DataLoader`** handles batching for us, and also **shuffles** the training data so that each **epoch** — each full pass over the training set — presents the images in a fresh random order.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

# Grab one batch to check the shapes:
images, labels = next(iter(train_loader))
print(f"One batch of images: {list(images.shape)}   # [batch, channels, height, width]")
print(f"One batch of labels: {list(labels.shape)}")

Always **look at your data** — many deep learning bugs are really data bugs:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i, 0], cmap="gray")   # images[i, 0]: i-th image, first (only) channel
    ax.set_title(f"label: {labels[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

---
### 🧠 Quick check 1

**Q:** Why did we set `shuffle=True` for the training loader but `shuffle=False` for the test loader?

<details><summary><b>Show answer</b></summary>

**Training:** the dataset is stored in an arbitrary order, and without shuffling the network would see the images in exactly the same sequence every epoch. Any ordering pattern (e.g. long runs of the same digit) biases the gradient updates; fresh random batches give more representative gradients and better training.

**Testing:** we only *measure* performance — no weights are updated — so the order cannot change the result. Keeping it fixed also makes evaluation reproducible.

</details>

---

---
# Part 2 — Convolutions: the building block of vision models

Could we not just feed the 784 pixels (28 × 28) straight into ordinary fully-connected layers? We could — but two things go wrong:

1. **It ignores spatial structure.** Flattening treats pixel (3, 4) and pixel (4, 4) as unrelated inputs, throwing away the very thing that makes an image an image.
2. **It doesn't scale.** A fully-connected layer from 784 pixels to just 128 units already needs ~100,000 weights. For a 224 × 224 colour photo, the same layer would need ~19 *million*.

A **convolutional layer** fixes both. It slides a small filter (also called a *kernel*), e.g. **3 × 3 weights**, across the image. At each position it computes a weighted sum of the pixels under the filter, producing one output pixel. The result is a **feature map** showing *where* in the image the pattern encoded by the filter occurs.

Two ideas make this powerful:

- **Local connectivity** — each output looks at a small neighbourhood, respecting spatial structure.
- **Weight sharing** — the *same* 9 weights are used at every position, so a pattern (an edge, a corner, a pen-stroke) is detected wherever it appears, and the layer needs only a handful of parameters.

Before CNNs, computer vision researchers designed such filters *by hand*. Here is a classic — the **Sobel filter**, which responds to vertical edges. Let's apply it to a digit with `F.conv2d` and see what it does:

In [ ]:
image, label = train_dataset[1]   # one digit image, shape [1, 28, 28]

# A hand-crafted 3x3 filter that responds to vertical edges.
# Shape [1, 1, 3, 3] = [output channels, input channels, height, width]:
sobel = torch.tensor([[-1., 0., 1.],
                      [-2., 0., 2.],
                      [-1., 0., 1.]]).reshape(1, 1, 3, 3)

with torch.no_grad():
    edges = F.conv2d(image.unsqueeze(0), sobel, padding=1)   # unsqueeze adds a batch dimension

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image[0], cmap="gray")
axes[0].set_title(f"input (a {label})")
axes[1].imshow(sobel[0, 0], cmap="gray")
axes[1].set_title("3x3 filter")
axes[2].imshow(edges[0, 0], cmap="gray")
axes[2].set_title("output: vertical edges")
for ax in axes:
    ax.axis("off")
plt.show()

### Under the microscope: how one output pixel is computed

The Sobel picture shows *what* a convolution does — now let's watch *how*, with numbers small enough to check by hand. The cell below builds a tiny 6 × 6 image (dark on the left, bright on the right) and slides the same Sobel filter across it. At each position the window stops and:

1. the 9 filter weights line up with the 9 pixels underneath,
2. each weight is multiplied by the pixel under it,
3. the 9 products are summed → **one pixel of the output**.

The red boxes mark one window position and the single output pixel it produces, and the printed line spells out that pixel's *entire* computation — there is nothing more to a convolution than this, repeated at every position.

**Edit `row` and `col`** (each 0 to 3) and re-run the cell to move the window. Two things worth noticing as you do:

- Over *flat* regions — all dark or all bright — the output is **0**; the filter responds only where its window straddles the boundary. That is what "detecting a vertical edge" means.
- The 6 × 6 input shrinks to a 4 × 4 output, because the window has to fit entirely inside the image. Surrounding the image with a border of zeros — the `padding=1` we used above — is what lets the output keep the input's size.

In [ ]:
tiny = torch.zeros(6, 6)
tiny[:, 3:] = 1.0                # a tiny test image: dark on the left, bright on the right

row, col = 1, 1                  # top-left corner of the window — edit me and re-run!
assert 0 <= row <= 3 and 0 <= col <= 3, "a 3x3 window on a 6x6 image: row and col must be 0..3"

with torch.no_grad():
    out = F.conv2d(tiny.reshape(1, 1, 6, 6), sobel).squeeze()   # no padding: 6x6 -> 4x4

# Spell out the multiply-and-sum for this window position:
window = tiny[row:row+3, col:col+3]
terms = [f"({sobel[0, 0, i, j]:g})×{window[i, j]:g}" for i in range(3) for j in range(3)]
print(f"output[{row},{col}] = " + " + ".join(terms) + f" = {(sobel[0, 0] * window).sum():g}")

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.6))
for ax, grid, title in zip(axes, [tiny, sobel[0, 0], out],
                           ["input  6×6", "filter  3×3", "output  4×4 (no padding)"]):
    ax.imshow(grid, cmap="gray")
    for i in range(grid.shape[0]):
        for j in range(grid.shape[1]):
            colour = "black" if grid[i, j] > grid.min() + (grid.max() - grid.min()) / 2 else "white"
            ax.text(j, i, f"{grid[i, j]:g}", ha="center", va="center", color=colour)
    ax.set_title(title)
    ax.axis("off")

# red boxes: the window on the input, and the one output pixel it produces
axes[0].add_patch(plt.Rectangle((col - .5, row - .5), 3, 3, edgecolor="red", facecolor="none", lw=2.5))
axes[2].add_patch(plt.Rectangle((col - .5, row - .5), 1, 1, edgecolor="red", facecolor="none", lw=2.5))
plt.show()

---
### 🔬 Kernel playground — design your own filter

Just **nine numbers** completely decide what a filter detects. So let's play with them — on a real digit this time. The next cell defines `show_convolution(kernel)`, which applies any 3 × 3 kernel to a digit (with `padding=1`, so the size is preserved) and shows the input, the kernel, and the resulting **feature map**. Because outputs can be negative, the feature map is shown in colour: **red = positive response, blue = negative, white ≈ 0**.

Four classic hand-crafted kernels are provided as presets — `"vertical edges"` (our Sobel from above), `"horizontal edges"`, `"blur"` and `"sharpen"`. **Look at each kernel's numbers and try to predict what it will do *before* running it.** Then experiment:

- Swap `"vertical edges"` for the other presets — do the feature maps match your predictions?
- Edit `my_kernel` and invent your own. What does a kernel of nine equal values do? All zeros except the centre?
- Multiply every value of an edge kernel by −1 — the colours flip. Why?
- Try other digits, e.g. `show_convolution(my_kernel, train_dataset[42][0])`.

Whatever you invent, you are exploring exactly the space the network will soon be searching for itself — nine numbers at a time.

In [ ]:
kernels = {
    "vertical edges":   [[-1., 0., 1.],
                         [-2., 0., 2.],
                         [-1., 0., 1.]],
    "horizontal edges": [[-1., -2., -1.],
                         [ 0.,  0.,  0.],
                         [ 1.,  2.,  1.]],
    "blur":             [[1/9, 1/9, 1/9],
                         [1/9, 1/9, 1/9],
                         [1/9, 1/9, 1/9]],
    "sharpen":          [[ 0., -1.,  0.],
                         [-1.,  5., -1.],
                         [ 0., -1.,  0.]],
}

def show_convolution(kernel, image=image):
    # Apply a 3x3 kernel to one digit image and show input | kernel | feature map.
    kernel = torch.as_tensor(kernel, dtype=torch.float32)
    with torch.no_grad():
        output = F.conv2d(image.unsqueeze(0), kernel.reshape(1, 1, 3, 3), padding=1)[0, 0]

    fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.3))
    axes[0].imshow(image[0], cmap="gray")
    axes[0].set_title("input")

    k = kernel.abs().max().item() or 1.0    # symmetric colour scale (the `or` guards an all-zero kernel)
    axes[1].imshow(kernel, cmap="RdBu_r", vmin=-k, vmax=k)
    for i in range(3):
        for j in range(3):
            colour = "white" if kernel[i, j].abs() > 0.6 * k else "black"
            axes[1].text(j, i, f"{kernel[i, j]:.2g}", ha="center", va="center", color=colour)
    axes[1].set_title("kernel")

    m = output.abs().max().item() or 1.0
    axes[2].imshow(output, cmap="RdBu_r", vmin=-m, vmax=m)
    axes[2].set_title("feature map (red +, blue −)")

    for ax in axes:
        ax.axis("off")
    plt.show()

# Try the presets first — predict, then run:
show_convolution(kernels["vertical edges"])

# ...then design your own! Edit the nine numbers and re-run:
my_kernel = [[ 0., -1.,  0.],
             [-1.,  4., -1.],
             [ 0., -1.,  0.]]
show_convolution(my_kernel)

---
### 🧠 Quick check 2

**Q:** The two edge kernels' nine values each sum to **0**, while the blur kernel's values sum to **1**. Imagine sliding a kernel over a perfectly *flat* patch of image, where every pixel has the same value $v$. What is the output pixel — and what does that tell you about kernels whose values sum to 0?

<details><summary><b>Show answer</b></summary>

Over a flat patch every one of the nine products is (weight $\times\, v$), so the output pixel is simply $v \times (\text{sum of the kernel's values})$. A kernel summing to **0** therefore outputs **exactly 0 on every flat region**, no matter how bright it is: the filter is blind to absolute brightness and responds only to *change* — edges, strokes, texture — which is precisely what a detector should do. The blur kernel sums to 1, so flat regions pass through unchanged and only the fine details get smoothed away.

</details>

---

**The key idea of deep learning:** nobody designs the filters any more. The filter values are the network's **weights**, and training *learns* them from data. Early layers typically discover edge- and blob-detectors much like Sobel; deeper layers combine them into detectors for curves, loops, and eventually whole digits.

Two more ingredients complete the picture:

- **ReLU** (`max(0, x)`) — a simple non-linearity applied after each layer: negative values become 0, positive ones pass through unchanged (on the playground's feature maps: keep the red, zero out the blue). Without it, a stack of convolutions would collapse mathematically into one single linear operation, no matter how deep. *(Try to convince yourself why!)*
- **Max-pooling** (`nn.MaxPool2d(2)`) — keeps the maximum of each 2 × 2 block, halving the width and height. This shrinks the computation and makes the features robust to small shifts of the input.

**Shape arithmetic** you will use constantly. Two knobs control a convolution's geometry: **padding** adds a border of $p$ zero-valued pixels around the input (so the filter can sit centred on the edge pixels), and **stride** is how many pixels the filter jumps between positions ($s = 1$ everywhere in this notebook). A convolution with kernel size $k$, padding $p$ and stride $s$ maps an input of size $H$ to output size

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

With $k=3, p=1, s=1$ (our usual setting) the size is **unchanged**; a 2 × 2 max-pool then **halves** it.

---
### 🧠 Quick check 3

**Q1:** A 28 × 28 image goes through a 3 × 3 convolution with `padding=1`, then a 2 × 2 max-pool. What is the output size?

**Q2:** How many learnable parameters does `nn.Conv2d(1, 16, kernel_size=3)` have? (Don't forget that each of the 16 filters also has a bias term.)

<details><summary><b>Show answer</b></summary>

**Q1:** The convolution keeps 28 × 28 (padding 1 exactly compensates the 3 × 3 kernel: $(28 + 2 - 3) + 1 = 28$); the pool then halves it to **14 × 14**.

**Q2:** Each filter has $1 \times 3 \times 3 = 9$ weights plus 1 bias, and there are 16 filters: $16 \times (9 + 1) = $ **160 parameters**. Compare that with the ~100,000 of a small fully-connected layer!

</details>

---

---
# Part 3 — Building a CNN

In PyTorch, a model is a class that inherits from **`nn.Module`** and defines two things:

- **`__init__`** — creates the layers (which hold the learnable weights);
- **`forward`** — says how data flows through those layers.

Our network stacks two *conv → ReLU → pool* blocks — each conv layer holding a whole bank of learnable 3 × 3 kernels, just like the ones from your playground — then flattens the feature maps into a vector and applies two fully-connected (`Linear`) layers to produce **10 scores, one per digit**:

```text
input                [B,  1, 28, 28]
Conv2d(1→16) + ReLU  [B, 16, 28, 28]     3x3 kernels, padding 1
MaxPool(2)           [B, 16, 14, 14]
Conv2d(16→32) + ReLU [B, 32, 14, 14]     3x3 kernels, padding 1
MaxPool(2)           [B, 32,  7,  7]
flatten              [B, 1568]           32 x 7 x 7 = 1568
Linear(1568→128) + ReLU
Linear(128→10)       [B, 10]             one score per digit class
```

The 10 outputs are raw, unnormalised scores called **logits** — they can be any real number, and bigger means "more likely this class". We deliberately do *not* apply softmax inside the model; you will see why in Part 4.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1 input channel  -> 16 feature maps
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 feature maps  -> 32 feature maps
        self.pool  = nn.MaxPool2d(2)                               # halves height and width
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)                            # 10 output classes

    def forward(self, x):                        # x: [B, 1, 28, 28]
        x = self.pool(F.relu(self.conv1(x)))     # -> [B, 16, 14, 14]
        x = self.pool(F.relu(self.conv2(x)))     # -> [B, 32, 7, 7]
        x = torch.flatten(x, start_dim=1)        # -> [B, 1568]  (keep the batch dimension!)
        x = F.relu(self.fc1(x))                  # -> [B, 128]
        return self.fc2(x)                       # -> [B, 10] logits

model = SimpleCNN().to(device)   # build it and move the weights to our device
print(model)

Before training anything, always **sanity-check the model**: push a dummy batch through and confirm the output shape, and count the parameters we are about to train.

In [ ]:
dummy = torch.randn(2, 1, 28, 28, device=device)   # a fake batch of 2 "images"
out = model(dummy)
print(f"Output shape: {list(out.shape)}   # [batch, classes] — looks right!")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

---
### 🧠 Quick check 4

**Q:** Why is the input size of `fc1` exactly `32 * 7 * 7`? And where does most of this network's ~207k parameters live?

<details><summary><b>Show answer</b></summary>

After two poolings the 28 × 28 image has shrunk to 7 × 7 (28 → 14 → 7), and the second conv layer produces 32 channels, so flattening gives a vector of $32 \times 7 \times 7 = 1568$ numbers — which is what `fc1` must accept.

The parameters are dominated by `fc1`: $1568 \times 128 + 128 = 200{,}832$ of the total 206,922 — about 97%! The two conv layers contribute only 4,800 parameters between them. This is typical: convolutions are very parameter-efficient, fully-connected layers are not.

</details>

---

---
# Part 4 — Training

Right now the network's weights are random numbers, so its predictions are guesses. **Training** means adjusting the weights, little by little, to make predictions match the labels on the training set. Beyond the data and model we already have, this needs exactly **three more things**:

### 1. A loss function

A single number measuring *how wrong* the predictions are — lower is better. Training is nothing more than minimising this number.

For classification, the standard choice is **cross-entropy loss** (`nn.CrossEntropyLoss`). Given the 10 logits, it (internally) applies **softmax** to turn them into probabilities that sum to 1, then takes the **negative log** of the probability assigned to the *correct* class. Confidently right → loss near 0; confidently wrong → large loss. This is also why our model outputs raw logits: `nn.CrossEntropyLoss` *expects logits* and does the softmax itself, in a numerically stable way.

PyTorch ships a whole catalogue of loss functions for different tasks — see the [`torch.nn` docs](https://docs.pytorch.org/docs/stable/nn.html#loss-functions). You will experiment with others in Exercise 2.

### 2. An optimiser

The rule for updating the weights. Backpropagation (`loss.backward()`) gives us the **gradient** of the loss with respect to every weight — the direction that would *increase* the loss — so the optimiser steps the other way:

$$w \leftarrow w - \eta \cdot \nabla_w \, \mathcal{L}$$

where $\eta$ is the **learning rate**, the most important hyperparameter in deep learning. Plain gradient descent is `optim.SGD`; we use **`optim.Adam`**, a variant that adapts the step size per weight and usually "just works" with $\eta = 10^{-3}$. Other choices live in [`torch.optim`](https://docs.pytorch.org/docs/stable/optim.html).

### 3. The training loop

Deep learning frameworks do *not* hide the loop from you — writing it yourself is normal PyTorch practice, and it is always the same five steps:

1. take a batch of data, move it to the device
2. **forward pass** — run the model to get predictions
3. compute the **loss**
4. **backward pass** — `loss.backward()` computes all gradients (after zeroing the old ones!)
5. **update** — `optimiser.step()` adjusts the weights

One full pass over the training set is called an **epoch**.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, loader, criterion, optimiser):
    # One full pass over the training data. Returns the per-batch losses.
    model.train()                    # put the model in training mode
    batch_losses = []

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)                  # step 1: batch onto the device
        labels = labels.to(device)

        logits = model(images)                      # step 2: forward pass
        loss = criterion(logits, labels)            # step 3: how wrong are we?

        optimiser.zero_grad()                       # step 4: clear old gradients...
        loss.backward()                             #         ...and compute new ones
        optimiser.step()                            # step 5: update the weights

        batch_losses.append(loss.item())            # .item(): tensor -> plain float, for logging
        if batch_idx % 100 == 0:
            print(f"   batch {batch_idx:3d}/{len(loader)}   loss: {loss.item():.4f}")

    return batch_losses

We also want to measure **accuracy**: the fraction of images classified correctly. The predicted class is simply the position of the *largest* logit (`argmax`). Note the two evaluation habits — `model.eval()` and `torch.no_grad()` — explained in the quick check below.

In [ ]:
def accuracy(model, loader):
    # Fraction of samples classified correctly.
    model.eval()                     # evaluation mode
    correct, total = 0, 0
    with torch.no_grad():            # we are only measuring — no gradients needed
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            predictions = logits.argmax(dim=1)          # index of the largest logit = predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total

Time to train — **2 epochs**, which takes roughly half a minute on a Colab GPU and a couple of minutes on a CPU. Before we start, check the untrained accuracy: with 10 classes, random guessing should give about 10%. Watch the loss fall as the network learns.

In [ ]:
print(f"Test accuracy BEFORE training: {accuracy(model, test_loader):.1%}   (random guessing ≈ 10%)\n")

n_epochs = 2
all_losses = []

for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1}/{n_epochs}")
    all_losses += train_one_epoch(model, train_loader, criterion, optimiser)
    print(f"   test accuracy: {accuracy(model, test_loader):.1%}\n")

From 10% to ~98–99% in two epochs. Let's plot the loss over the training steps — the raw per-batch loss is noisy (every batch is different), so we overlay a moving average to see the trend:

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(all_losses, alpha=0.3, label="per-batch loss")

window = 25
smoothed = np.convolve(all_losses, np.ones(window) / window, mode="valid")
plt.plot(np.arange(window - 1, len(all_losses)), smoothed, label=f"moving average ({window} batches)")

plt.xlabel("training step (batch)")
plt.ylabel("cross-entropy loss")
plt.legend()
plt.show()

---
### 🧠 Quick check 5

**Q1:** What would happen if we forgot `optimiser.zero_grad()` in the loop?

**Q2:** Why do we call `model.eval()` and wrap evaluation in `torch.no_grad()`?

<details><summary><b>Show answer</b></summary>

**Q1:** PyTorch *accumulates* gradients into `.grad` on every `backward()` call. Without zeroing, each step would apply the sum of all gradients so far — the updates grow and point in stale directions, and training typically diverges. (Forgetting `zero_grad()` is one of the most common PyTorch bugs!)

**Q2:** `model.eval()` switches layers that behave differently at train and test time (dropout, batch-norm) into their deterministic test behaviour — our current model has none, but it's an essential habit. `torch.no_grad()` tells PyTorch not to build the computation graph needed for backpropagation, making evaluation faster and lighter on memory since we will never call `backward()` here.

</details>

---

---
# Part 5 — Evaluation: beyond a single number

Accuracy is one number — useful, but it hides *what kind* of mistakes the model makes. A **confusion matrix** shows, for each true digit (rows), how often each digit was predicted (columns). A perfect model puts everything on the diagonal.

In [ ]:
confusion = torch.zeros(10, 10, dtype=torch.int64)

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        predictions = logits.argmax(dim=1).cpu()
        for true, pred in zip(labels, predictions):
            confusion[true, pred] += 1

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(confusion, cmap="Blues")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("predicted digit")
ax.set_ylabel("true digit")
ax.set_title("Confusion matrix (test set)")
for i in range(10):
    for j in range(10):
        colour = "white" if confusion[i, j] > confusion.max() // 2 else "black"
        ax.text(j, i, confusion[i, j].item(), ha="center", va="center", color=colour, fontsize=8)
plt.show()

Even more instructive is looking at the **individual mistakes**. Some are genuinely ambiguous scrawls that humans would also get wrong; others reveal systematic weaknesses of the model.

In [ ]:
wrong_images, wrong_true, wrong_pred = [], [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        predictions = logits.argmax(dim=1).cpu()
        mistakes = predictions != labels               # boolean mask over the batch
        wrong_images.append(images[mistakes])
        wrong_true.append(labels[mistakes])
        wrong_pred.append(predictions[mistakes])

wrong_images = torch.cat(wrong_images)
wrong_true = torch.cat(wrong_true)
wrong_pred = torch.cat(wrong_pred)
print(f"The model got {len(wrong_images)} of {len(test_dataset)} test images wrong.")

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.axis("off")
    if i < len(wrong_images):
        ax.imshow(wrong_images[i, 0], cmap="gray")
        ax.set_title(f"true: {wrong_true[i].item()}  pred: {wrong_pred[i].item()}", fontsize=9)
plt.tight_layout()
plt.show()

---
### 🧠 Quick check 6

**Q:** Look at the off-diagonal entries of your confusion matrix. Which pairs of digits does the model confuse most, and can you guess why?

<details><summary><b>Show answer</b></summary>

The exact numbers vary run to run, but the usual suspects are **4 ↔ 9**, **3 ↔ 5**, **7 ↔ 2** and **8 ↔ 3** — pairs that share most of their pen strokes and differ in small details (whether the top of the 4/9 is closed, the top-left curve of 3/5, ...). The model's mistakes mirror the visual similarity structure of the digits themselves — sloppy handwriting sits genuinely between two classes.

</details>

---

---
# Part 6 — Exercises ✏️

Now it is your turn. Everything you need has appeared above — and note that our helper `train_one_epoch(...)` takes the model, loss and optimiser as *arguments* (and `accuracy(...)` takes any model and loader), so you can reuse both for every experiment below.

Solutions are at the very end of the notebook, but **write your own attempt first** — that is where the learning happens.

*Short on time? Prioritise Exercise 1; in Exercise 2 the label-smoothing option is a one-line change; Exercise 3 makes a good take-home.*

## Exercise 1 — Build a deeper CNN

`SimpleCNN` has **two** convolutional layers. Build a `DeeperCNN` with **three** conv blocks:

- three blocks of *conv (3 × 3, padding 1) → ReLU → max-pool (2 × 2)*, with channels going **1 → 16 → 32 → 64**;
- then flatten, `Linear(? → 128)`, ReLU, and `Linear(128 → 10)`.

Your main job is to work out the `?` — the flattened feature size after the third block. Use the shape arithmetic from Part 2 and mind the rounding!

**Before you train it, make a prediction:** will `DeeperCNN` have *more* or *fewer* parameters than `SimpleCNN`'s 206,922? Deeper should mean bigger... shouldn't it?

<details><summary><b>Hint (the rule)</b></summary>

Each 3 × 3 conv with padding 1 keeps the spatial size; each 2 × 2 max-pool halves it, **rounding down**. Check carefully what 7 becomes after pooling!

</details>

<details><summary><b>Full answer (only if you're stuck)</b></summary>

28 → 14 → 7 → **3** (7 halves to 3, not 3.5 — the pool simply drops the leftover row and column), so the flattened size is $64 \times 3 \times 3 = 576$.

</details>

In [ ]:
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define three conv layers (channels 1->16, 16->32, 32->64,
        #       each kernel_size=3, padding=1), one pooling layer,
        #       and two linear layers (? -> 128 and 128 -> 10).
        raise NotImplementedError("define the layers, then delete this line")

    def forward(self, x):
        # TODO: three times (conv -> relu -> pool), then flatten, fc1 -> relu, fc2.
        raise NotImplementedError("write the forward pass, then delete this line")

In [ ]:
# This cell trains your DeeperCNN. It will just print a reminder until
# you have completed the TODOs above.
try:
    deeper_model = DeeperCNN().to(device)

    # Sanity check before training:
    out = deeper_model(torch.randn(2, 1, 28, 28, device=device))
    assert list(out.shape) == [2, 10], f"expected output [2, 10], got {list(out.shape)}"

    n_simple = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_deeper = sum(p.numel() for p in deeper_model.parameters() if p.requires_grad)
    print(f"SimpleCNN: {n_simple:,} parameters | DeeperCNN: {n_deeper:,} parameters\n")

    deeper_optimiser = torch.optim.Adam(deeper_model.parameters(), lr=1e-3)
    for epoch in range(2):
        print(f"Epoch {epoch + 1}/2")
        train_one_epoch(deeper_model, train_loader, criterion, deeper_optimiser)
        print(f"   test accuracy: {accuracy(deeper_model, test_loader):.1%}\n")

except NotImplementedError as todo:
    print(f"⚠️ Not finished yet — {todo}")

## Exercise 2 — Try a different loss function

The loss function defines *what the network actually optimises*, and PyTorch provides a whole catalogue of them in [`torch.nn`](https://docs.pytorch.org/docs/stable/nn.html#loss-functions). Pick **at least one** alternative, train a *fresh* `SimpleCNN` with it, and compare the final test accuracy with the ~98–99% we got from plain cross-entropy. Some suggestions, roughly in order of difficulty:

1. **`nn.CrossEntropyLoss(label_smoothing=0.1)`** — instead of asking for 100% confidence in the true class, the target becomes a blend of 90% one-hot label and 10% *uniform distribution over all ten classes* — so the true digit gets probability 0.91 and every other digit 0.01. A widely-used regularisation trick — one argument, big literature.
2. **`nn.NLLLoss`** — the *negative log-likelihood* loss. It expects **log-probabilities**, not logits, so you must apply `F.log_softmax(..., dim=1)` at the end of your model's forward pass. In fact, `CrossEntropyLoss` *is exactly* `log_softmax` + `NLLLoss` — a good way to prove that to yourself.
3. **`nn.MSELoss`** *(stretch)* — mean squared error is a *regression* loss; to use it for classification you need to turn each label into a **one-hot vector** (`F.one_hot(labels, num_classes=10).float()`) and compare it with the model's (softmaxed) outputs. It works — but notice *how much slower* it learns, and think about why it is a poor fit for classification.

⚠️ One caveat: loss *values* from different loss functions are not comparable with each other — compare models by **accuracy**.

In [ ]:
# TODO 1: pick a loss function from torch.nn (see the link above).
new_criterion = ...        # e.g. nn.CrossEntropyLoss(label_smoothing=0.1)

# TODO 2 (only if your chosen loss needs it): some losses expect the model to
# output something other than raw logits — you may need to modify the model.
loss_model = SimpleCNN().to(device)

if new_criterion is ...:
    print("⚠️ Choose a loss function above (TODO 1), then re-run this cell.")
else:
    loss_optimiser = torch.optim.Adam(loss_model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(2):
        print(f"Epoch {epoch + 1}/2")
        losses += train_one_epoch(loss_model, train_loader, new_criterion, loss_optimiser)
        print(f"   test accuracy: {accuracy(loss_model, test_loader):.1%}\n")

## Exercise 3 (optional stretch) — Optimisers and learning rates

If you have time left, investigate the *optimiser* side of training. Train a fresh `SimpleCNN` for one epoch with each of these settings and compare the loss curves and accuracies:

- `torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)` — the classic;
- `torch.optim.Adam(..., lr=1e-3)` — our default;
- `torch.optim.Adam(..., lr=1.0)` — a learning rate that is far too large;
- `torch.optim.Adam(..., lr=1e-6)` — far too small.

You should see the whole story of the learning rate in four curves: too big *diverges or oscillates*, too small *crawls*, just right *converges quickly*.

In [ ]:
# Free experimentation cell — copy the training pattern from Exercise 2.
# Tip: collect each run's `losses` list under a different name, then plot them
# on the same axes to compare.

---
# Wrap-up: Practical 1

In one hour you have covered the complete deep learning workflow, end to end:

| Ingredient | What we used |
|---|---|
| **Data** | MNIST via `torchvision`, transforms, `DataLoader` batching |
| **Model** | a CNN: stacked *conv → ReLU → pool* blocks + linear classifier head |
| **Loss** | cross-entropy (and friends, in Exercise 2) |
| **Optimiser** | Adam / SGD from `torch.optim` |
| **Loop** | forward → loss → backward → step, over mini-batches |
| **Evaluation** | held-out test set, accuracy, confusion matrix, error inspection |

Everything larger — from ResNets to the vision models behind self-driving cars — is this same recipe with bigger ingredients.

**What we deliberately left out** (your next topics to explore): validation splits and hyperparameter tuning, data augmentation, batch normalisation and dropout, learning-rate schedules, modern architectures (ResNets, vision transformers), and transfer learning from pre-trained models. Several of these — validation splits and transfer learning in particular — show up in **Practical 2, coming up next**.

### Where to go next

- [dl-pytorch](https://github.com/atapour/dl-pytorch) — Amir Atapour-Abarghouei's teaching examples (GANs, VAEs, transformers and more), which inspired this notebook
- [Official PyTorch tutorials](https://docs.pytorch.org/tutorials/) — start with the 60-minute blitz
- [3Blue1Brown's neural network series](https://www.3blue1brown.com/topics/neural-networks) — beautiful visual intuition for backpropagation
- [CS231n](https://cs231n.github.io/) — Stanford's convolutional networks course notes
- [BMVA](https://www.bmva.org/) — the British Machine Vision Association

---

# Solutions

Try the exercises first! Then compare with the reference implementations below.

<details><summary><b>Solution — Exercise 1: DeeperCNN</b></summary>

```python
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 3 * 3, 128)   # 28 -> 14 -> 7 -> 3, so 64*3*3 = 576
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # [B, 16, 14, 14]
        x = self.pool(F.relu(self.conv2(x)))   # [B, 32, 7, 7]
        x = self.pool(F.relu(self.conv3(x)))   # [B, 64, 3, 3]
        x = torch.flatten(x, start_dim=1)      # [B, 576]
        x = F.relu(self.fc1(x))
        return self.fc2(x)
```

**The parameter surprise:** DeeperCNN has only **98,442** parameters — *less than half* of SimpleCNN's 206,922, despite being deeper! The third conv block costs a modest 18,496 parameters but shrinks the feature map from 7 × 7 to 3 × 3, cutting `fc1` from $1568 \times 128$ down to $576 \times 128$ weights. This is a recurring theme in CNN design: **depth is cheap; big fully-connected layers are expensive.** Accuracy after 2 epochs is typically ~99%, similar to or slightly better than SimpleCNN.

</details>

<details><summary><b>Solution — Exercise 2: different loss functions</b></summary>

**Option 1 — label smoothing** is a one-line change:

```python
new_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
loss_model = SimpleCNN().to(device)
```

Accuracy stays in the ~98–99% range. Note the loss no longer approaches 0 — with smoothed targets, even a perfect model has non-zero loss, so its *value* is not comparable to the plain cross-entropy runs.

**Option 2 — NLLLoss** needs the model to output log-probabilities. The tidy way is a subclass that reuses `SimpleCNN`:

```python
class SimpleCNNLogSoftmax(SimpleCNN):
    def forward(self, x):
        return F.log_softmax(super().forward(x), dim=1)

new_criterion = nn.NLLLoss()
loss_model = SimpleCNNLogSoftmax().to(device)
```

Training behaves *identically* to `CrossEntropyLoss` — because it is the same function, just split into two steps. You can verify the equivalence directly:

```python
logits = torch.randn(4, 10)
labels = torch.tensor([1, 0, 3, 9])
print(nn.CrossEntropyLoss()(logits, labels))
print(nn.NLLLoss()(F.log_softmax(logits, dim=1), labels))   # same number
```

(And `accuracy` still works unchanged: `log_softmax` is monotonic, so the `argmax` is the same.)

**Option 3 — MSE on one-hot targets.** `train_one_epoch` calls `criterion(logits, labels)` with integer labels, so wrap the conversion inside a custom criterion function — a plain Python function works fine in place of an `nn` module:

```python
def mse_criterion(logits, labels):
    targets = F.one_hot(labels, num_classes=10).float()
    return F.mse_loss(torch.softmax(logits, dim=1), targets)

new_criterion = mse_criterion
loss_model = SimpleCNN().to(device)
```

It reaches a decent accuracy but learns visibly more slowly. MSE penalises a confident wrong answer only quadratically (the error per output is at most 1), whereas cross-entropy's $-\log p$ penalty grows *without bound* as the probability of the true class goes to 0 — giving much stronger gradients exactly where the model is most wrong. That, plus its clean probabilistic interpretation, is why cross-entropy is the default for classification.

</details>

<details><summary><b>Solution — Exercise 3: optimisers and learning rates</b></summary>

```python
runs = {
    "SGD, lr=0.01, momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam, lr=1e-3":          lambda p: torch.optim.Adam(p, lr=1e-3),
    "Adam, lr=1.0":           lambda p: torch.optim.Adam(p, lr=1.0),
    "Adam, lr=1e-6":          lambda p: torch.optim.Adam(p, lr=1e-6),
}

plt.figure(figsize=(9, 4))
for name, make_optimiser in runs.items():
    torch.manual_seed(42)                      # same starting weights for a fair comparison
    run_model = SimpleCNN().to(device)
    run_losses = train_one_epoch(run_model, train_loader, criterion,
                                 make_optimiser(run_model.parameters()))
    smoothed = np.convolve(run_losses, np.ones(25) / 25, mode="valid")
    plt.plot(smoothed, label=f"{name}  (acc {accuracy(run_model, test_loader):.1%})")

plt.xlabel("training step")
plt.ylabel("loss (moving average)")
plt.yscale("log")
plt.legend()
plt.show()
```

Typical outcome: Adam at `1e-3` and SGD with momentum both converge nicely; Adam at `1.0` jumps around a large loss and never really learns; Adam at `1e-6` decreases *painfully* slowly. The learning rate is the first thing to tune when training misbehaves.

</details>

---

*Created for the BMVA Summer School. Inspired by, and gratefully acknowledging, Amir Atapour-Abarghouei's [dl-pytorch](https://github.com/atapour/dl-pytorch) materials.*

---

---

# 🔀 Interlude — From Classification to Segmentation

Congratulations on finishing Practical 1! You now have a network that answers *"**what** digit is in this image?"* with a single label per image. Practical 2 asks a finer-grained question: *"**which pixels** belong to what?"* — and everything you have just learned carries over directly:


A few practical notes before you dive in:

- **Fresh setup ahead.** Practical 2 re-imports everything it needs and *redefines* names from Practical 1 — `train_dataset`, `test_dataset`, `train_loader`, `train_one_epoch`, … — for the new task. That is intentional. It does mean that once you have started Practical 2, going back to re-run a Practical 1 training cell requires re-running Practical 1's definition cells first.
- **Part numbering restarts** at Part 0 below — the two practicals keep their original structure.
- **Bigger data.** Practical 2 downloads the Oxford-IIIT Pet dataset (~800 MB) in its Part 2 — kick that cell off and read on while it downloads.
- **GPU matters from here on.** Training an FCN is much heavier than MNIST. If you are stuck on CPU, Practical 2's Part 0 explains which knobs to shrink.

Onwards — from one label per image to a label for every pixel. 🐱🐶


# Practical 2 — Fully Convolutional Networks for Semantic Segmentation

## BMVA Summer School — Practical Session 2 of 2

---

Welcome back! In Practical 1 you built a CNN that assigns **one
label to a whole image**. Here we go a step further: **semantic segmentation** assigns a label to *every
pixel*. We'll take an ImageNet classifier apart and turn it into a **Fully Convolutional Network (FCN)** —
the architecture that first made per-pixel prediction with plain CNNs practical — and train it on real
images of cats and dogs.

The **core path** (Parts 0–8) gets you a working, trained FCN-32s model end to end. Parts 9–10, which add a
second model (FCN-16s) and compare the two, are an **optional extension** if time allows.

### How this notebook works

- Sections marked **✏️ Exercise** contain a skeleton with `# TODO` markers. Try to fill them in yourself first.
- Every exercise is immediately followed by a **✅ Reference solution** cell. The rest of the notebook always
  uses the reference solution, so it still runs top-to-bottom even if you skip an exercise — you can always
  come back and compare your attempt against it.
- **🧠 Quick check** boxes ask a short question about what you have just seen — think first, then click to
  reveal the answer.
- Parts 9–10 are an **optional extension** — skip straight to the Wrap-up if you're short on time.

### What's inside

**Core:**

| Part | Topic |
|:---|:---|
| 0 | Setup |
| 1 | What is semantic segmentation? |
| 2 | Dataset: Oxford-IIIT Pet |
| 3 | Building block: transposed convolution |
| 4 | From classifier to FCN |
| 5 | Implement FCN-32s |
| 6 | Loss and evaluation metrics |
| 7 | Training loop |
| 8 | Visualise predictions |

**Optional extension:**

| Part | Topic |
|:---|:---|
| 9 | FCN-16s: adding a skip connection |
| 10 | Comparing FCN-32s and FCN-16s with metrics |

This notebook needs `torch`, `torchvision`, `matplotlib`, `numpy` and `tqdm` — all pre-installed on Google
Colab. A GPU runtime is recommended (`Runtime > Change runtime type > GPU`); training also works on CPU if
you shrink `IMG_SIZE` and `TRAIN_SUBSET_SIZE`.

### Prerequisites

- Comfortable with PyTorch `nn.Module`, `Dataset`/`DataLoader`, and basic CNNs (conv, pooling, ReLU) — covered
  in Practical 1 above.
- No prior segmentation experience required.

---
# Part 0 — Setup

Run this on **Google Colab with a GPU runtime** (`Runtime > Change runtime type > GPU`) if possible. If you
want to train on CPU instead, shrink `IMG_SIZE` (Part 2) and `TRAIN_SUBSET_SIZE` (Part 7) first.

In [ ]:
# Uncomment the line below if you are running this on a fresh Colab / local environment.
# Requires torchvision >= 0.15 (for the OxfordIIITPet dataset and the VGG16_Weights API used below).
# !pip install -q "torch>=2.0" "torchvision>=0.15" matplotlib tqdm

import random
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from torchvision.datasets import OxfordIIITPet
from tqdm.auto import tqdm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


---
# Part 1 — What is semantic segmentation?

Given an image, **semantic segmentation** assigns a class label to *every pixel*, rather than a single label
to the whole image (classification) or a bounding box per object (detection). The following example
illustrates the difference.

![Image classification vs. object detection vs. image segmentation, illustrated on an image of two sheep](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/classification_detection_segmentation.png)

*Classification answers "what is a sheep?" with one label for the whole image. Object detection adds "where
is it?" with a box per object. Segmentation goes further and answers "which pixels belong to which object?"
with a full pixel-wise mask. (Source: [dida.do](https://dida.do), used here for teaching purposes.)*

The three tasks differ in what the model actually outputs:

| Task | Output |
|:---|:---|
| Classification | one label for the whole image, e.g. `"sheep"` |
| Object detection | a box + label per object, e.g. `[x1,y1,x2,y2, "sheep"]` |
| Semantic segmentation | a label **per pixel**, e.g. a `(H, W)` map of `{pet, background, border}` |

**Why bother labelling individual pixels?** Because often it isn't enough to know *what* is in an image — you also
want to know *where* it is, what *shape* it has, and which pixels belong to which object. Framing the problem
this way lets a neural network output a dense, pixel-wise mask of the image instead of a single tag,
giving a much finer-grained understanding of the scene than classification or detection alone. This is why
semantic segmentation shows up in applications like medical imaging (outlining a tumour or organ), self-driving
cars (separating road, pedestrians, and other vehicles pixel by pixel), and satellite imaging (mapping land
use or building footprints).

---
# Part 2 — Dataset: Oxford-IIIT Pet

To explore semantic segmentation hands-on, we use the [Oxford-IIIT Pet dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/)
via `torchvision.datasets.OxfordIIITPet` — a 37-category dog and cat dataset with roughly 200 images per class
(~7,400 images total, ~800 MB to download), with large variations in scale, pose, and lighting. Every image
comes with a **trimap** segmentation mask where every pixel takes one of three values:

| Raw trimap value | Meaning |
|:---|:---|
| 1 | Pet (foreground) |
| 2 | Background |
| 3 | Border / undefined |

`OxfordIIITPet` ships two official splits: `trainval` (about 3,680 images) and `test` (about 3,669 images),
loaded below as `train_pet_raw` and `test_pet_raw`.

![Example images and trimap annotations from the Oxford-IIIT Pet dataset](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/pet_annotations.jpg)

*Example images and their trimap annotations, from the dataset's official page
([robots.ox.ac.uk/~vgg/data/pets](https://www.robots.ox.ac.uk/~vgg/data/pets/)). Notice how the annotation
tightly follows each animal's outline, with a thin "don't care" border region between foreground and
background — this is exactly the 3-class trimap ({pet, background, border}) we load and train on below.*

In [ ]:
DATA_ROOT = "./data"
IMG_SIZE = 128  # shrink to e.g. 96 if you're on CPU and want faster iterations

train_pet_raw = OxfordIIITPet(
    root=DATA_ROOT, split="trainval", target_types="segmentation", download=True
)
test_pet_raw = OxfordIIITPet(
    root=DATA_ROOT, split="test", target_types="segmentation", download=True
)

print(f"train/val images: {len(train_pet_raw)}")
print(f"test images:      {len(test_pet_raw)}")

image, trimap = train_pet_raw[0]
print(f"image size: {image.size}, mode: {image.mode}")
print(f"trimap size: {trimap.size}, mode: {trimap.mode}")
print(f"unique trimap values: {sorted(set(np.array(trimap).ravel().tolist()))}")

### Building a dataset loader

Straight out of the box, `OxfordIIITPet` hands us an `(image, trimap)` pair — full-size and untouched. But as we know, neural networks are fussy: every image in a batch must be the same size, and pixel values need to fall within a range the model expects. So before training starts, our dataset loader quietly takes care of three small chores:

1. **Resize the image and the mask to the same square size, but not the same way.** The image can be blurred
   a bit while resizing and still look fine. The mask can't: blend label `0` into label `1` and you get `0.5`,
   which isn't a class! So the mask gets resized with **nearest-neighbour** interpolation instead — it just
   copies the closest label rather than averaging.
2. **Normalise the image's pixel values** using ImageNet's mean/std. Since our segmentation model uses a pretrained VGG16 backbone that was trained on ImageNet, it expects input images to be normalised using the same statistics.
3. **Relabel the trimap** from `{1, 2, 3}` to `{0, 1, 2}`, since PyTorch expects class indices to start
   counting at zero.

The `SegmentationDataset` loader below bundles all of this up, so everything downstream only ever sees tidy
`(image, mask)` tensor pairs.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class SegmentationDataset(Dataset):
    # Loads OxfordIIITPet items, jointly resizing image + mask into label tensors.
    def __init__(self, base_dataset, img_size=IMG_SIZE):
        self.base_dataset = base_dataset
        self.img_size = img_size
        self.image_transform = transforms.Compose(
            [
                transforms.Resize((img_size, img_size)),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ]
        )
        self.mask_resize = transforms.Resize(
            (img_size, img_size), interpolation=transforms.InterpolationMode.NEAREST
        )

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, trimap = self.base_dataset[idx]
        image = self.image_transform(image)

        trimap = self.mask_resize(trimap)
        trimap = torch.from_numpy(np.array(trimap, dtype=np.int64))
        mask = trimap - 1  # {1,2,3} -> {0,1,2}

        return image, mask


CLASS_NAMES = ["pet", "background", "border"]
NUM_CLASSES = len(CLASS_NAMES)

train_dataset = SegmentationDataset(train_pet_raw)
test_dataset = SegmentationDataset(test_pet_raw)

image, mask = train_dataset[0]
print(f"image tensor: {image.shape}, dtype {image.dtype}")
print(f"mask tensor:  {mask.shape}, dtype {mask.dtype}, unique values {mask.unique().tolist()}")

### Visualise a few image samples

Always look at your data before writing a single line of model code.

In [ ]:
def denormalize(img_tensor):
    # Undo ImageNet normalisation for display. img_tensor: (3, H, W)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    img = img_tensor.cpu() * std + mean
    return img.clamp(0, 1).permute(1, 2, 0).numpy()


def show_samples(dataset, n=4, seed=SEED):
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), n)

    fig, axes = plt.subplots(2, n, figsize=(3 * n, 6))
    for col, idx in enumerate(indices):
        image, mask = dataset[idx]
        axes[0, col].imshow(denormalize(image))
        axes[0, col].set_title(f"image #{idx}")
        axes[0, col].axis("off")

        axes[1, col].imshow(mask.numpy(), cmap="viridis", vmin=0, vmax=NUM_CLASSES - 1)
        axes[1, col].set_title("mask (pet / bg / border)")
        axes[1, col].axis("off")
    plt.tight_layout()
    plt.show()


show_samples(train_dataset)

---
# Part 3 — Building block: transposed convolution

A classifier CNN shrinks an image down to a small spatial grid as it goes deeper — each pooling / strided
conv layer reduces resolution, with the amount set by its kernel size and stride. VGG16 (which we'll use later)
has 5 max-pools of `2x2`, stride 2, so a `128x128` input becomes a `4x4` grid (`128 / 2**5 = 4`). For
segmentation we need to go back **up** to the original resolution. `nn.ConvTranspose2d` (a "transposed" or
"learned upsampling" convolution) is one way to do this.

### Basic operation

Ignoring channels for now, consider the basic transposed convolution with stride 1 and no padding. Given an
$n_h \times n_w$ input tensor and a $k_h \times k_w$ kernel, we build the output like this: for each element
of the input, multiply the kernel by that element's value to get a $k_h \times k_w$ patch, then add that
patch onto an output canvas of size $(n_h + k_h - 1) \times (n_w + k_w - 1)$, placed at the position matching
where that element sits in the input. Summing the (overlapping) patches from every input element gives the
final output.

As an example, the figure below illustrates how transposed convolution with a $2 \times 2$ kernel is computed
for a $2 \times 2$ input tensor.

![Transposed convolution with a 2x2 kernel applied to a 2x2 input tensor](https://d2l.ai/_images/trans_conv.svg)

*(Source: [d2l.ai, "14.10. Transposed Convolution"](https://d2l.ai/chapter_computer-vision/transposed-conv.html), used here for teaching purposes.)*

### Padding and stride

Padding and stride work the opposite way here compared to a regular convolution. A regular conv pads the
*input* before sliding the kernel; a transposed conv pads the *output* instead — `padding=1`, for example,
trims the first and last row and column off the result.

Stride also acts on the output: instead of controlling how far the kernel slides over the input, it controls
how far apart the patches (from the basic operation above) land on the output canvas. Using the same input
and kernel as before, moving from stride 1 to stride 2 spreads the patches further apart, so the output grows
in both height and width, as shown below.

![Transposed convolution with a 2x2 kernel and stride 2, applied to a 2x2 input tensor](https://d2l.ai/_images/trans_conv_stride2.svg)

*(Source: [d2l.ai, "14.10. Transposed Convolution"](https://d2l.ai/chapter_computer-vision/transposed-conv.html), used here for teaching purposes.)*

### Output size

Combining padding and stride, the output size for a 1-D transposed conv is:

```
out = (in - 1) * stride - 2 * padding + kernel_size
```

With `stride=1` and `padding=0` this reduces to `out = in + kernel_size - 1`, matching the canvas size
$(n_h + k_h - 1) \times (n_w + k_w - 1)$ from the basic operation above. Increasing `stride` grows the output
(as in the padding-and-stride figure above), while increasing `padding` shrinks it by trimming rows/columns
off the output.

Let's see it in action on a tiny toy tensor. PyTorch's conv layers expect 4-D tensors shaped
`(batch_size, channels, height, width)`, so the `1x1x2x2` example below is a single `2x2` feature map (batch
size 1, 1 channel) — not a literal image, just a small grid to make the operation easy to see by eye.

In [ ]:
toy = torch.arange(1, 5, dtype=torch.float32).view(1, 1, 2, 2)  # a 2x2 "feature map"
print("input:\n", toy.squeeze().numpy())

upsample = nn.ConvTranspose2d(
    in_channels=1, out_channels=1, kernel_size=2, stride=2, bias=False
)
with torch.no_grad():
    upsample.weight.fill_(1.0)  # fixed weights, just to make the effect obvious

out = upsample(toy)
print(f"\noutput shape: {tuple(out.shape)}  (we doubled H and W)")
print("output:\n", out.squeeze().detach().numpy())

---
### 🧠 Quick check 1

**Q:** Without running any code, what output shape would `nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1)` produce for a `1x1x8x8` input? Use the formula above.

<details><summary><b>Show answer</b></summary>

Using `out = (in - 1) * stride - 2 * padding + kernel_size`:

`out = (8 - 1) * 2 - 2 * 1 + 4 = 14 - 2 + 4 = 16`

So the output shape is `(1, 1, 16, 16)` — run the cell below to check.

</details>

---


In [ ]:
check_layer = nn.ConvTranspose2d(1, 1, kernel_size=4, stride=2, padding=1)
check_out = check_layer(torch.zeros(1, 1, 8, 8))
print(f"actual output shape: {tuple(check_out.shape)}")

---
# Part 4 — From classifier to FCN

Now it's time to build a segmentation model of our own. We will use the **Fully Convolutional Network (FCN)**
of Long, Shelhamer & Darrell, "Fully Convolutional Networks for Semantic Segmentation", CVPR 2015 — the paper
that popularised turning classifiers into dense, per-pixel predictors using only convolutional layers.

The FCN paper's key idea: take an ImageNet classifier (e.g. VGG16), and

1. **Keep VGG16's convolutional feature extractor** (`vgg16.features`), which has already learned useful visual features.
2. **Throw away the fully-connected classifier head**, since it collapses spatial information into a single vector, which is exactly what we *don't* want.
3. **Replace it with a `1x1` convolution**, which maps each spatial location's feature vector directly to
   `NUM_CLASSES` scores. A `1x1` conv is mathematically equivalent to applying the same linear classifier
   independently at every spatial location. This is the "fully convolutional" trick.
4. **Upsample** the resulting coarse, low-resolution score map back to the input resolution so we get one
   prediction per pixel.

![Pipeline of a VGG-style backbone (image -> conv1 -> pool1 -> ... -> pool5 -> conv6-7) followed by a single 32x upsampling step to produce the FCN-32s prediction](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/fcn32s_pipeline.png)

*(Adapted from Fig. 3 of Long, J., Shelhamer, E., & Darrell, T. (2015). Fully Convolutional Networks for
Semantic Segmentation. CVPR 2015. [arXiv:1411.4038](https://arxiv.org/abs/1411.4038), used here for teaching
purposes.)*

The figure above (the FCN-32s branch of the original paper's architecture diagram) shows exactly this
pipeline: VGG16's `features` has 5 max-pooling stages, and each `pool` stage halves the spatial resolution
(the grid inside each box gets coarser) until `pool5`, where a `128x128` input becomes a `4x4` grid of
512-channel feature vectors (downsampled by a **factor of 32**). This coarse grid is then upsampled 32x in one
shot back to the input resolution. This naive, single-shot upsampling is called **FCN-32s**.

Let's load VGG16's convolutional backbone and check where its max-pooling layers sit.

In [ ]:
def make_vgg16_backbone(pretrained=True):
    # Returns vgg16.features, i.e. only the convolutional part (no avgpool / classifier).
    weights = models.VGG16_Weights.DEFAULT if pretrained else None
    vgg16 = models.vgg16(weights=weights)
    return vgg16.features


# Check: confirm where each max-pool sits inside vgg16.features.
# We rely on these indices later to tap intermediate feature maps for skip connections.
backbone_probe = make_vgg16_backbone(pretrained=False)
for i, layer in enumerate(backbone_probe):
    if isinstance(layer, nn.MaxPool2d):
        print(f"MaxPool2d at index {i}")


For upsampling, the original paper uses a *learned* `nn.ConvTranspose2d`, initialised to start out
mathematically equivalent to plain bilinear interpolation, then fine-tuned during training. This gives the
best of both worlds: a sensible, non-random starting point, plus room to learn something better than fixed
bilinear upsampling. Below, `bilinear_kernel` computes those initial weights, and `make_bilinear_upsample`
loads them into a real `ConvTranspose2d` layer. *(Source: [d2l.ai, "Fully Convolutional Networks"](https://d2l.ai/chapter_computer-vision/fcn.html), used here
for teaching purposes.)*

In [ ]:
def bilinear_kernel(in_channels, out_channels, kernel_size):
    # Returns ConvTranspose2d weights that reproduce bilinear upsampling exactly.
    factor = (kernel_size + 1) // 2
    center = factor - 1 if kernel_size % 2 == 1 else factor - 0.5
    coords = torch.arange(kernel_size)
    filt_1d = 1 - torch.abs(coords - center) / factor
    filt = filt_1d.view(-1, 1) * filt_1d.view(1, -1)

    weight = torch.zeros((in_channels, out_channels, kernel_size, kernel_size))
    weight[range(in_channels), range(out_channels)] = filt
    return weight


def make_bilinear_upsample(num_classes, stride):
    # A ConvTranspose2d that upsamples by `stride`x, initialised to bilinear interpolation
    # (kernel_size=2*stride, padding=stride//2 makes the output exactly `stride` times the input size).
    kernel_size = 2 * stride
    padding = stride // 2
    layer = nn.ConvTranspose2d(
        num_classes, num_classes, kernel_size=kernel_size, stride=stride,
        padding=padding, bias=False,
    )
    with torch.no_grad():
        layer.weight.copy_(bilinear_kernel(num_classes, num_classes, kernel_size))
    return layer


# Check: compare our bilinear-initialised ConvTranspose2d against F.interpolate.
toy_scores = torch.randn(1, 1, 4, 4)
bilinear_layer = make_bilinear_upsample(num_classes=1, stride=32)
learned_out = bilinear_layer(toy_scores)
interp_out = F.interpolate(toy_scores, scale_factor=32, mode="bilinear", align_corners=False)
diff = (learned_out - interp_out).abs()

# Away from the border (a strip one stride wide on each edge) they agree almost exactly, because
# both compute the same interior bilinear weights. Right at the border they diverge, because
# ConvTranspose2d's zero-padding and F.interpolate's edge handling disagree there.
border = 32
interior_diff = diff[:, :, border:-border, border:-border]
print(f"interior max abs diff: {interior_diff.max().item():.6f}")
print(f"full-image max abs diff (border included): {diff.max().item():.6f}")


---
# Part 5 — Implement FCN-32s

**✏️ Exercise:** complete `FCN32sExercise.forward`. You need to:

1. Run `x` through `self.backbone` to get the final feature map (shape `(N, 512, H/32, W/32)`).
2. Apply `self.classifier` (a `1x1` conv) to get per-location class scores (shape `(N, num_classes, H/32, W/32)`).
3. Upsample those scores back to the *original* input resolution `(H, W)` with `self.upsample` — a learned
   `ConvTranspose2d`, initialised by `make_bilinear_upsample` above to start out equivalent to bilinear
   interpolation.

`self.backbone`, `self.classifier`, and `self.upsample` are already created for you in `__init__`.

In [ ]:
class FCN32sExercise(nn.Module):
    def __init__(self, num_classes, pretrained_backbone=True):
        super().__init__()
        self.backbone = make_vgg16_backbone(pretrained=pretrained_backbone)
        self.classifier = nn.Conv2d(512, num_classes, kernel_size=1)
        self.upsample = make_bilinear_upsample(num_classes, stride=32)

    def forward(self, x):
        input_size = x.shape[-2:]

        # TODO: 1. extract features with self.backbone
        # features = ...

        # TODO: 2. classify each spatial location with self.classifier
        # scores = ...

        # TODO: 3. upsample `scores` back to `input_size` with self.upsample
        # out = ...

        raise NotImplementedError("Fill in the three TODOs above")

        # Already implemented for you: handles inputs whose size isn't an exact
        # multiple of 32, so the output always matches input_size exactly.
        if out.shape[-2:] != input_size:
            out = F.interpolate(out, size=input_size, mode="bilinear", align_corners=False)
        return out


### ✅ Reference solution

In [ ]:
class FCN32s(nn.Module):
    def __init__(self, num_classes, pretrained_backbone=True):
        super().__init__()
        self.backbone = make_vgg16_backbone(pretrained=pretrained_backbone)
        self.classifier = nn.Conv2d(512, num_classes, kernel_size=1)
        self.upsample = make_bilinear_upsample(num_classes, stride=32)

    def forward(self, x):
        input_size = x.shape[-2:]

        features = self.backbone(x)                  # (N, 512, H/32, W/32)
        scores = self.classifier(features)            # (N, num_classes, H/32, W/32)
        out = self.upsample(scores)                    # (N, num_classes, H, W)
        if out.shape[-2:] != input_size:
            # Only triggers if H/W aren't exact multiples of 32 — falls back to bilinear
            # interpolation so the output shape always matches the input exactly.
            out = F.interpolate(out, size=input_size, mode="bilinear", align_corners=False)
        return out


### Shape check

Before training anything, always verify the output shape matches what you expect on a dummy batch.


In [ ]:
model = FCN32s(num_classes=NUM_CLASSES).to(DEVICE)

with torch.no_grad():
    dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    dummy_output = model(dummy_input)

expected_shape = (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
assert tuple(dummy_output.shape) == expected_shape, (
    f"expected {expected_shape}, got {tuple(dummy_output.shape)}"
)
print(f"OK — output shape: {tuple(dummy_output.shape)}")

---
# Part 6 — Loss and evaluation metrics

We use pixel-wise cross-entropy as the loss to train the model, and pixel accuracy plus mean IoU to evaluate
how well it segments images.

- **Loss:**
  - **Pixel-wise cross-entropy:** Let $p_{ij}$ be the model's predicted probability of the **correct** class
    for pixel $(i,j)$ (after the softmax). The loss for that pixel is
    $$\ell_{ij} = -\log p_{ij},$$
    and averaging over all pixels gives
    $$\mathcal{L} = -\frac{1}{H W}\sum_{i,j} \log p_{ij}.$$
    `nn.CrossEntropyLoss` computes exactly this (extended over the batch) when given logits of shape
    `(N, C, H, W)` and integer targets of shape `(N, H, W)`. It treats each pixel as an independent
    classification problem.

- **Evaluation:**
  - **Pixel accuracy:** Let $\hat{y}_{ij}$ be the predicted class and $y_{ij}$ the ground-truth class for
    pixel $(i,j)$. Pixel accuracy is the fraction of correctly classified pixels:
    $$\text{Pixel Accuracy} = \frac{1}{H W}\sum_{i,j} \mathbb{1}[\hat{y}_{ij} = y_{ij}].$$
    It is easy to compute, but can be misleading on datasets dominated by one class (e.g. mostly background).
  - **Mean IoU (Intersection over Union):** For each class $c$, let $P_c$ be the set of pixels predicted as
    class $c$, and $G_c$ the set of ground-truth pixels belonging to class $c$. Then
    $$\text{IoU}_c = \frac{|P_c \cap G_c|}{|P_c \cup G_c|}, \qquad \text{mIoU} = \frac{1}{C}\sum_{c=1}^{C} \text{IoU}_c.$$
    Unlike pixel accuracy, mIoU penalises both false positives and false negatives, making it the standard
    metric for semantic segmentation.

**✏️ Exercise:** implement `pixel_accuracy` and `mean_iou`.

In [ ]:
def pixel_accuracy_exercise(preds, targets):
    # preds, targets: (N, H, W) integer tensors of class indices
    # TODO: return the fraction of pixels where preds == targets, as a python float
    raise NotImplementedError


def mean_iou_exercise(preds, targets, num_classes):
    # preds, targets: (N, H, W) integer tensors of class indices
    # TODO: for each class c, compute the intersection and union of predicted vs. ground-truth pixels
    #       skip classes with union == 0 (not present in this batch)
    #       return the average IoU across the remaining classes, as a python float
    raise NotImplementedError

### ✅ Reference solution

In [ ]:
def pixel_accuracy(preds, targets):
    correct = (preds == targets).float().sum()
    total = torch.numel(targets)
    return (correct / total).item()


def mean_iou(preds, targets, num_classes):
    ious = []
    for c in range(num_classes):
        pred_c = preds == c
        target_c = targets == c
        intersection = (pred_c & target_c).sum().item()
        union = (pred_c | target_c).sum().item()
        if union == 0:
            continue  # class not present in this batch — skip rather than count as 0
        ious.append(intersection / union)
    return float(np.mean(ious)) if ious else 0.0


# quick unit test on a toy example
_p = torch.tensor([[0, 0, 1], [1, 1, 1]])
_t = torch.tensor([[0, 1, 1], [1, 1, 0]])
print(f"toy pixel accuracy: {pixel_accuracy(_p, _t):.3f} (expect 0.667)")
print(f"toy mean IoU:       {mean_iou(_p, _t, num_classes=2):.3f}")

---
# Part 7 — Training loop

For a lab session we train on a **subset** of the data for a handful of epochs — enough to clearly see the
model learn, without needing a long run. Increase `TRAIN_SUBSET_SIZE` and `NUM_EPOCHS` if you have a Colab
GPU and time to spare.

We hold out a small **validation** subset from the training pool for epoch-by-epoch monitoring. Unlike the
training and validation subsets, the **test** split is run through the model in full, to report a final
unbiased number. By default this means training on 2,000 images, validating on 200 images each epoch, and
evaluating on the full ~3,669-image test split.

Let's set the key hyperparameters, then randomly sample a train/validation split from `train_dataset` and
wrap all three splits (train/val/test) in `DataLoader`s.

In [ ]:
TRAIN_SUBSET_SIZE = 2000  # set to len(train_dataset) - VAL_SUBSET_SIZE for a full run
VAL_SUBSET_SIZE = 200  # held out from the trainval pool, used only for epoch-by-epoch monitoring
BATCH_SIZE = 16
NUM_EPOCHS = 9
LEARNING_RATE = 1e-4

rng = random.Random(SEED)
subset_indices = rng.sample(
    range(len(train_dataset)), min(TRAIN_SUBSET_SIZE + VAL_SUBSET_SIZE, len(train_dataset))
)
train_indices = subset_indices[:TRAIN_SUBSET_SIZE]
val_indices = subset_indices[TRAIN_SUBSET_SIZE:]

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(
    f"training on {len(train_subset)} images, validating each epoch on {len(val_subset)} images, "
    f"final test set has {len(test_dataset)} images (evaluated in full)"
)

Let's define the training building blocks: `train_one_epoch` runs one pass of forward/backward/optimizer
steps over the training data, `evaluate` computes loss and metrics without updating weights, and `fit` ties
them together into the full multi-epoch training loop.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(loader, desc="train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, num_classes):
    model.eval()
    running_loss, accs, ious = 0.0, [], []
    for images, masks in tqdm(loader, desc="eval", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        logits = model(images)
        loss = criterion(logits, masks)
        running_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1)
        accs.append(pixel_accuracy(preds, masks))
        ious.append(mean_iou(preds, masks, num_classes))

    return {
        "loss": running_loss / len(loader.dataset),
        "pixel_acc": float(np.mean(accs)),
        "mean_iou": float(np.mean(ious)),
    }


def fit(model, train_loader, val_loader, num_epochs=NUM_EPOCHS, lr=LEARNING_RATE):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, num_epochs + 1):
        start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_metrics = evaluate(model, val_loader, criterion, NUM_CLASSES)
        elapsed = time.time() - start

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "pixel_acc": val_metrics["pixel_acc"],
            "mean_iou": val_metrics["mean_iou"],
        })
        print(
            f"epoch {epoch}/{num_epochs} | train_loss {train_loss:.4f} | "
            f"val_loss {val_metrics['loss']:.4f} | pixel_acc {val_metrics['pixel_acc']:.3f} | "
            f"mean_iou {val_metrics['mean_iou']:.3f} | {elapsed:.1f}s"
        )

    return history


Let's instantiate `FCN32s` and train it with `fit`, storing the per-epoch train/val loss and metrics in
`fcn32s_history`.

In [ ]:
fcn32s_model = FCN32s(num_classes=NUM_CLASSES).to(DEVICE)
fcn32s_history = fit(fcn32s_model, train_loader, val_loader)


---
# Part 8 — Visualise predictions

Numbers only tell half the story — always look at the actual predicted masks.

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, n=4, seed=SEED):
    model.eval()
    rng = random.Random(seed)
    indices = rng.sample(range(len(dataset)), n)

    fig, axes = plt.subplots(3, n, figsize=(3 * n, 9))
    for col, idx in enumerate(indices):
        image, mask = dataset[idx]
        logits = model(image.unsqueeze(0).to(DEVICE))
        pred = logits.argmax(dim=1).squeeze(0).cpu()

        axes[0, col].imshow(denormalize(image))
        axes[0, col].set_title(f"image #{idx}")
        axes[0, col].axis("off")

        axes[1, col].imshow(mask.numpy(), cmap="viridis", vmin=0, vmax=NUM_CLASSES - 1)
        axes[1, col].set_title("ground truth")
        axes[1, col].axis("off")

        axes[2, col].imshow(pred.numpy(), cmap="viridis", vmin=0, vmax=NUM_CLASSES - 1)
        axes[2, col].set_title("prediction")
        axes[2, col].axis("off")
    plt.tight_layout()
    plt.show()


show_predictions(fcn32s_model, test_dataset)

---
### 🧠 Quick check 2

**Q:** Look closely at the predicted masks near the pet's outline. Why does FCN-32s tend to produce blobby, low-detail boundaries?

<details><summary><b>Show answer</b></summary>

Because the entire prediction is computed from a single `4x4` grid (for a `128x128` input) — each of those 16 cells has to speak for a `32x32` block of the original image — and the 32x upsample has no way to add detail that isn't already present in that coarse grid. Part 9 fixes this by fusing in an earlier, higher-resolution feature map.

</details>

---

---
# Part 9 (optional extension) — FCN-16s: adding a skip connection

*Short on time? This is a good place to stop — skip to the Wrap-up at the end. Everything from
here on builds a second model and compares it to FCN-32s.*

FCN-32s' final feature map (`pool5`) is downsampled by a factor of 32 relative to the input. The preceding
max-pooling layer (`pool4`) is downsampled by only a factor of 16 — twice the resolution, and still contains
fairly high-level semantic features (unlike very early layers, which mostly encode edges/textures).
**FCN-16s** exploits this by incorporating `pool4`'s finer-grained features with `pool5`'s coarse ones,
producing more detailed segmentation predictions:

1. Compute the coarse `pool5`-based class scores, same as FCN-32s.
2. Upsample those scores **2x** to match `pool4`'s resolution, using a learned,
   bilinear-initialised `ConvTranspose2d` (the same `make_bilinear_upsample` helper from Part 4/5).
3. Apply a *separate* `1x1` conv classifier to the `pool4` features to get a second set of class scores at
   the same resolution.
4. **Fuse** the two score maps by adding them together (the "skip connection"). The fine-grained `pool4`
   scores correct the coarse `pool5` prediction with local detail.
5. Upsample the fused score map by the remaining 16x back to the input resolution, again with a learned,
   bilinear-initialised `ConvTranspose2d`.

The figure below (the same paper diagram as Part 4's, this time its FCN-16s row) shows this visually: the
`pool4` features feed forward directly, while the `pool5` path is upsampled 2x and *added* to them (the
overlapping boxes represent this sum) before the final 16x upsample. For comparison, the FCN-32s output from
Part 4 is shown alongside — notice it skips the fusion step entirely and just upsamples the coarse `pool5`
scores straight to full resolution.

![Pipeline for FCN-16s: the same VGG-style backbone as FCN-32s, but pool4 features are fused with the 2x-upsampled pool5/conv7 scores before the final 16x upsample, producing a sharper prediction than FCN-32s' direct 32x upsample](https://raw.githubusercontent.com/Adaptive-Intelligence-Lab/BMVA-summer-school/main/assets/fcn16s_pipeline.png)

*(Adapted from Fig. 3 of Long, J., Shelhamer, E., & Darrell, T. (2015). Fully Convolutional Networks for
Semantic Segmentation. CVPR 2015. [arXiv:1411.4038](https://arxiv.org/abs/1411.4038), used here for teaching
purposes.)*

Now let's build **FCN-16s**.

**✏️ Exercise:** complete `FCN16sExercise.forward`. `self.backbone` is now split into two pieces:
`self.backbone_to_pool4` (everything up to and including `pool4`) and `self.pool4_to_pool5` (the rest, up to
`pool5`), so you can grab the intermediate `pool4` activations.

In [ ]:
# From the MaxPool2d index printout in Part 4: pool3 -> index 16, pool4 -> index 23, pool5 -> index 30.
POOL4_LAYER_INDEX = 23  # backbone[:24] ends right after pool4
POOL5_LAYER_INDEX = 30  # backbone[:31] ends right after pool5 (== full backbone)


class FCN16sExercise(nn.Module):
    def __init__(self, num_classes, pretrained_backbone=True):
        super().__init__()
        full_backbone = make_vgg16_backbone(pretrained=pretrained_backbone)
        self.backbone_to_pool4 = full_backbone[: POOL4_LAYER_INDEX + 1]
        self.pool4_to_pool5 = full_backbone[POOL4_LAYER_INDEX + 1 : POOL5_LAYER_INDEX + 1]

        self.score_pool4 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.score_pool5 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.upsample_2x = make_bilinear_upsample(num_classes, stride=2)
        self.upsample_16x = make_bilinear_upsample(num_classes, stride=16)

    def forward(self, x):
        input_size = x.shape[-2:]

        # TODO: 1. get pool4 features with self.backbone_to_pool4
        # pool4_features = ...

        # TODO: 2. get pool5 features with self.pool4_to_pool5, applied to pool4_features
        # pool5_features = ...

        # TODO: 3. classify both feature maps with self.score_pool4 and self.score_pool5
        # score4 = ...
        # score5 = ...

        # TODO: 4. upsample score5 2x with self.upsample_2x
        # score5_up = ...

        raise NotImplementedError("Fill in TODOs 1-4 above")

        # Already implemented for you: handles score5_up and score4 not lining up exactly
        # (can happen when IMG_SIZE isn't an exact multiple of 32).
        if score5_up.shape[-2:] != score4.shape[-2:]:
            score5_up = F.interpolate(
                score5_up, size=score4.shape[-2:], mode="bilinear", align_corners=False
            )

        # TODO: 5. add score5_up to score4 — this is the "skip connection"
        # fused = ...

        # TODO: 6. upsample fused 16x with self.upsample_16x
        # out = ...

        raise NotImplementedError("Fill in TODOs 5-6 above")

        # Already implemented for you: same fix, this time for the final output vs input_size.
        if out.shape[-2:] != input_size:
            out = F.interpolate(out, size=input_size, mode="bilinear", align_corners=False)
        return out


### ✅ Reference solution

In [ ]:
class FCN16s(nn.Module):
    def __init__(self, num_classes, pretrained_backbone=True):
        super().__init__()
        full_backbone = make_vgg16_backbone(pretrained=pretrained_backbone)
        self.backbone_to_pool4 = full_backbone[: POOL4_LAYER_INDEX + 1]
        self.pool4_to_pool5 = full_backbone[POOL4_LAYER_INDEX + 1 : POOL5_LAYER_INDEX + 1]

        self.score_pool4 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.score_pool5 = nn.Conv2d(512, num_classes, kernel_size=1)
        self.upsample_2x = make_bilinear_upsample(num_classes, stride=2)
        self.upsample_16x = make_bilinear_upsample(num_classes, stride=16)

    def forward(self, x):
        input_size = x.shape[-2:]

        pool4_features = self.backbone_to_pool4(x)             # downsampled by 16
        pool5_features = self.pool4_to_pool5(pool4_features)    # downsampled by 32

        score4 = self.score_pool4(pool4_features)                # (N, C, H/16, W/16)
        score5 = self.score_pool5(pool5_features)                # (N, C, H/32, W/32)

        score5_up = self.upsample_2x(score5)
        if score5_up.shape[-2:] != score4.shape[-2:]:
            score5_up = F.interpolate(
                score5_up, size=score4.shape[-2:], mode="bilinear", align_corners=False
            )
        fused = score4 + score5_up                                # skip connection

        out = self.upsample_16x(fused)
        if out.shape[-2:] != input_size:
            out = F.interpolate(out, size=input_size, mode="bilinear", align_corners=False)
        return out


Now let's instantiate `FCN16s`, check its output shape, then train it on the same data as FCN-32s so the two are directly comparable.

In [ ]:
fcn16s_model = FCN16s(num_classes=NUM_CLASSES).to(DEVICE)

with torch.no_grad():
    dummy_output = fcn16s_model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE))
assert tuple(dummy_output.shape) == (2, NUM_CLASSES, IMG_SIZE, IMG_SIZE)
print("shape check OK")

fcn16s_history = fit(fcn16s_model, train_loader, val_loader)


Let's visualise FCN-16s' predictions.

In [ ]:
show_predictions(fcn16s_model, test_dataset)

---
# Part 10 (optional extension) — Comparing FCN-32s and FCN-16s with metrics

In [ ]:
# Final, unbiased numbers: this is the *first and only* time we touch `test_loader`.
final_criterion = nn.CrossEntropyLoss()
fcn32s_test_metrics = evaluate(fcn32s_model, test_loader, final_criterion, NUM_CLASSES)
fcn16s_test_metrics = evaluate(fcn16s_model, test_loader, final_criterion, NUM_CLASSES)

rows = [
    ("FCN-32s", fcn32s_test_metrics),
    ("FCN-16s", fcn16s_test_metrics),
]

print(f"{'model':<10} {'test_loss':>10} {'pixel_acc':>10} {'mean_iou':>10}")
for name, m in rows:
    print(f"{name:<10} {m['loss']:>10.4f} {m['pixel_acc']:>10.3f} {m['mean_iou']:>10.3f}")


---
### 🧠 Quick check 3

**Q:** With only `TRAIN_SUBSET_SIZE` images and `NUM_EPOCHS` epochs, the performance gap between FCN-32s and FCN-16s may look modest in the aggregate metrics above. Where should you look to see the real benefit of the skip connection?

<details><summary><b>Show answer</b></summary>

Compare the *visualised* predictions from Part 8 (FCN-32s) and Part 9 (FCN-16s) side by side, not just the mean IoU numbers. FCN-16s should show visibly sharper, more accurate boundaries around the pet, even where mean IoU looks similar between the two. This illustrates an important point: aggregate metrics don't always reflect improvements in boundary quality.
    
</details>

---

---
# Wrap-up

Well done! In this practical you covered the complete recipe for turning an image classifier into a dense per-pixel segmentation model, **FCN-32s**:

| Ingredient | What we used |
|:---|:---|
| **Data** | Oxford-IIIT Pet via `torchvision`, joint image/mask resizing, a train/val/test split |
| **Backbone** | VGG16's convolutional features, with the classifier head removed |
| **Decoder head** | a `1x1` conv (a per-pixel linear classifier) + a learned, bilinear-initialised `ConvTranspose2d` |
| **Loss** | pixel-wise cross-entropy |
| **Evaluation** | pixel accuracy and mean IoU |
| **FCN-16s** *(optional)* | a skip connection fusing an earlier, higher-resolution feature map to sharpen boundaries |

**What we left out** (your next topics to explore): FCN-8s and beyond, alternative segmentation architectures
(U-Net, DeepLab, Mask R-CNN), modern transformer-based segmentation models, and the Segment Anything Model (SAM).

### Where to go next

- [dl-pytorch](https://github.com/atapour/dl-pytorch) — Amir Atapour-Abarghouei's teaching examples (GANs, VAEs, transformers and more)
- [d2l.ai's FCN chapter](https://d2l.ai/chapter_computer-vision/fcn.html) — the same bilinear-initialisation trick, worked through in more detail
- [BMVA](https://www.bmva.org/) — the British Machine Vision Association

---
# References

- Long, J., Shelhamer, E., & Darrell, T. (2015). *Fully Convolutional Networks for Semantic Segmentation.*
  CVPR 2015. https://arxiv.org/abs/1411.4038
- Zhang, A., Lipton, Z. C., Li, M., & Smola, A. J. (2023). *Dive into Deep Learning.* Cambridge University
  Press. https://d2l.ai
- Parkhi, O. M., Vedaldi, A., Zisserman, A., & Jawahar, C. V. (2012). *Cats and Dogs* (Oxford-IIIT Pet
  Dataset). CVPR 2012. https://www.robots.ox.ac.uk/~vgg/data/pets/
- `torchvision` documentation: https://pytorch.org/vision/stable/index.html

---

*Created for the BMVA Summer School as the second practical, following Practical 1 above. Builds on Long, Shelhamer & Darrell (2015) and the FCN chapter of Zhang, Lipton, Li & Smola's* Dive into Deep Learning.
